### **TASK 1**

In [1]:
import re
import pdfplumber

pdf_path = "sample.pdf"

In [2]:
# (a) Extract text page by page
pages = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        pages.append(page.extract_text() or "")

print("Total pages:", len(pages))

Total pages: 4


In [3]:
# (b) Merge and clean text
all_text = " ".join(pages)

clean_text = re.sub(r"[\d\W_]+", " ", all_text)
tokens = clean_text.lower().split()

print("Total tokens:", len(tokens))

Total tokens: 1323


In [4]:
# (c) Word frequency with stopwords
from collections import Counter

stopwords = {
    "the", "is", "a", "an", "and", "or", "of", "to", "in",
    "on", "for", "with", "as", "by", "this", "that", "are",
    "was", "were", "it", "from", "at", "be"
}

filtered_tokens = [word for word in tokens if word not in stopwords]

freq = Counter(filtered_tokens)

print("Top 10 words:")
for word, count in freq.most_common(10):
    print(word, ":", count)

Top 10 words:
nlp : 26
used : 24
token : 23
pipeline : 20
text : 19
like : 18
matters : 15
where : 15
why : 14
d : 14


In [5]:
# (d) Search keyword page by page
keyword = input("Enter keyword: ").lower()

found = False

for page_number, page_text in enumerate(pages, 1):
    sentences = re.split(r'(?<=[.!?])\s+', page_text)

    for sentence in sentences:
        if keyword in sentence.lower():
            print(f"Page {page_number}: {sentence.strip()}")
            found = True

if not found:
    print("Keyword not found.")


Page 1: Why it matters: Sometimes we do not need the whole match - we only need one small part of it (for example,
only the year from a date).
Page 2: Part B: Pattern-Based Extraction (using re.findall())
Concept: re.findall(pattern, text) finds all matches of a pattern in a piece of text and returns them as a list.
Page 2: Email Extraction
import re
pattern = r'[\w.-]+@[\w-]+\.[a-zA-Z]{2,}'
re.findall(pattern, text)
Why it matters: We often need to automatically find email addresses, for example for contact lists or spam
filtering.
Page 2: Order ID Extraction
Example format: ORD-2024-00123
pattern = r'ORD-\d{4}-\d{5}'
Why it matters: Business emails and documents often contain specific IDs that need to be pulled out
automatically.
Page 3: Tokenization means breaking text into small pieces called tokens.
Page 3: Tokens are usually words and punctuation
marks.
Page 4: Quick Recap Table (Pipeline Order)
Step Topic Pipeline Stage
1 Regex Fundamentals Preprocessing / cleaning
2 Pattern-Bas

### **TASK 2**

In [6]:
import pdfplumber

pdf_path = "sample.pdf"

In [7]:
# (a)
pages_data = []

with pdfplumber.open(pdf_path) as pdf:
    for number, page in enumerate(pdf.pages, 1):
        text = page.extract_text() or ""
        words = text.split()
        
        pages_data.append({
            "page_number": number,
            "raw_text": text,
            "word_count": len(words),
            "char_count": len(text.replace(" ", ""))
        })

print("First 2 entries:")
print(pages_data[:2])

First 2 entries:
[{'page_number': 1, 'raw_text': 'Part A: Regex Fundamentals (Foundation)\nA1. What is Regex?\nRegex (Regular Expression) is a special pattern language. It is used to find specific pieces of text, such as emails,\nphone numbers, or dates, inside a big block of text.\nWhy it matters: Raw text (emails, chats, documents) contains useful information mixed with a lot of other text.\nWithout regex, we would have to search everything by hand, which is slow and full of mistakes.\nWhere it is used in NLP pipeline: Regex is used in the preprocessing step. Before any NLP model works on\nthe data, we clean the text or pull out specific information (like emails or phone numbers) using regex.\nA2. Character Classes\nCharacter classes tell regex what type of character to look for.\nSymbol Meaning\n\\d Any digit (0-9)\n\\w Any word character (letters, digits, underscore)\n\\s Space, tab, or new line\n[a-z] Any lowercase letter from a to z\n[A-Z0-9] A custom set - capital letters or dig

In [8]:
# (b)
non_zero_pages = [p for p in pages_data if p["word_count"] > 0]

if non_zero_pages:
    highest = max(non_zero_pages, key=lambda x: x["word_count"])
    lowest = min(non_zero_pages, key=lambda x: x["word_count"])

    print("Highest word count page:", highest["page_number"])
    print(highest["raw_text"])

    print("Lowest non-zero word count page:", lowest["page_number"])

    average = sum(p["word_count"] for p in pages_data) / len(pages_data)
    print("Average word count:", round(average, 2))
else:
    print("No pages contain words.")

Highest word count page: 1
Part A: Regex Fundamentals (Foundation)
A1. What is Regex?
Regex (Regular Expression) is a special pattern language. It is used to find specific pieces of text, such as emails,
phone numbers, or dates, inside a big block of text.
Why it matters: Raw text (emails, chats, documents) contains useful information mixed with a lot of other text.
Without regex, we would have to search everything by hand, which is slow and full of mistakes.
Where it is used in NLP pipeline: Regex is used in the preprocessing step. Before any NLP model works on
the data, we clean the text or pull out specific information (like emails or phone numbers) using regex.
A2. Character Classes
Character classes tell regex what type of character to look for.
Symbol Meaning
\d Any digit (0-9)
\w Any word character (letters, digits, underscore)
\s Space, tab, or new line
[a-z] Any lowercase letter from a to z
[A-Z0-9] A custom set - capital letters or digits
Why it matters: These are the basic b

In [9]:
# (c)
class PhraseNotFoundError(Exception):
    pass

phrase = input("Enter search phrase: ").lower()
found = False

for page in pages_data:
    count = page["raw_text"].lower().count(phrase)
    
    if count > 0:
        print(f"Page {page['page_number']}: {count} occurrence(s)")
        found = True

try:
    if not found:
        raise PhraseNotFoundError("Phrase not found in any page.")
except PhraseNotFoundError as e:
    print(e)

Page 1: 1 occurrence(s)
Page 2: 6 occurrence(s)
Page 3: 3 occurrence(s)
Page 4: 1 occurrence(s)


In [10]:
# (d)
import csv

with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["page_number", "word_count", "char_count"])
    
    for page in pages_data:
        writer.writerow([
            page["page_number"],
            page["word_count"],
            page["char_count"]
        ])

# Re-read the file
with open("pdf_summary.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    
    for row in reader:
        print(row)

['page_number', 'word_count', 'char_count']
['1', '381', '1792']
['2', '266', '1540']
['3', '329', '1825']
['4', '300', '1758']


### **TASK 3**

In [11]:
# (a)
import pdfplumber
import re

with pdfplumber.open("sample.pdf") as pdf:
    text = "\n".join(page.extract_text() or "" for page in pdf.pages)

before = len(text)

# Remove URLs
text = re.sub(r"https?://\S+|www\.\S+", "", text)

# Remove digits
text = re.sub(r"\d+", "", text)

# Keep only alphabets, spaces and full stops
text = re.sub(r"[^A-Za-z\s.]", "", text)

# Collapse spaces
text = re.sub(r"\s+", " ", text)

after = len(text)

print("Characters before:", before)
print("Characters after:", after)

Characters before: 8052
Characters after: 7569


In [12]:
# (b)
text = text.lower()

sentences = re.split(r"[.!?]", text)

print("Sentence count before:", len(sentences))

sentences = [s.strip() for s in sentences if s.strip()]

print("Sentence count after:", len(sentences))

Sentence count before: 110
Sentence count after: 109


In [13]:
# (c)
import nltk

nltk.download("stopwords")

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

processed_sentences = []

for sentence in sentences:
    tokens = sentence.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    if len(tokens) >= 5:
        processed_sentences.append(" ".join(tokens))

print("Before filter:", len(sentences))
print("After filter:", len(processed_sentences))

print("\nSample sentences:")
for sentence in processed_sentences[:3]:
    print(sentence)

Before filter: 109
After filter: 64

Sample sentences:
regex regex regular expression special pattern language
used find specific pieces text emails phone numbers dates inside big block text
matters raw text emails chats documents contains useful information mixed lot text


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hadi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
# (d)
with open("nlp_ready_output.txt", "w", encoding="utf-8") as file:
    for sentence in processed_sentences:
        file.write(sentence + "\n")

with open("nlp_ready_output.txt", "r", encoding="utf-8") as file:
    lines = [line.strip() for line in file if line.strip()]

print("Total lines:", len(lines))

if lines:
    print("Longest sentence:", max(lines, key=len))
    print("Shortest sentence:", min(lines, key=len))

Total lines: 64
Longest sentence: quick recap table pipeline order step topic pipeline stage regex fundamentals preprocessing cleaning patternbased extraction information extraction word tokenization text tokens token attributes feature engineering filtering sentence segmentation tokens sentencelevel structure order matters real nlp system text first cleaned important information extracted tokenized check properties token finally text organized sentences
Shortest sentence: part c word tokenization c


### **TASK 4**

In [26]:
from docx import Document

doc = Document("sample.docx")

In [27]:


paragraphs = []

for paragraph in doc.paragraphs:
    paragraphs.append({
        "text": paragraph.text,
        "style": paragraph.style.name
    })

print("Total paragraphs:", len(paragraphs))

styles = {p["style"] for p in paragraphs}
print("Unique styles:", styles)

Total paragraphs: 79
Unique styles: {'Heading 2', 'Normal', 'Heading 1'}


In [28]:
# (b)
headings = []
body = []
other = []

for p in paragraphs:
    style = p["style"]
    
    if style.startswith("Heading"):
        headings.append(p)
    elif style == "Normal" or "Body" in style:
        body.append(p)
    else:
        other.append(p)

print("Headings:", len(headings))
print("Body:", len(body))
print("Other/Empty:", len(other))

Headings: 15
Body: 64
Other/Empty: 0


In [29]:
# (c)
qualifying = [
    p for p in body
    if len(p["text"].split()) > 5
]

if qualifying:
    highest = max(qualifying, key=lambda p: len(p["text"].split()))
    lowest = min(qualifying, key=lambda p: len(p["text"].split()))

    mean_words = sum(
        len(p["text"].split()) for p in qualifying
    ) / len(qualifying)

    print("Highest word count:", highest["text"])
    print("Lowest word count above 5:", lowest["text"])
    print("Mean word count:", round(mean_words, 2))

Highest word count: This order matters because in a real NLP system, text is first cleaned and important information is extracted, then it is tokenized, then we check the properties of each token, and finally the text is organized into sentences. After this, later steps like POS tagging, NER, or model training can begin. 
Lowest word count above 5: NLTK tokenization is simple, rule-based, and fast. 
Mean word count: 21.19


In [30]:
# (d)
from collections import defaultdict
from pprint import pprint

style_data = defaultdict(lambda: {"count": 0, "texts": []})

for p in paragraphs:
    style = p["style"]
    style_data[style]["count"] += 1
    style_data[style]["texts"].append(p["text"])

for style in style_data:
    if len(style_data[style]["texts"]) > 2:
        style_data[style]["texts"] = style_data[style]["texts"][:2]

sorted_styles = dict(
    sorted(
        style_data.items(),
        key=lambda x: x[1]["count"],
        reverse=True
    )
)

pprint(sorted_styles)

{'Heading 1': {'count': 5,
               'texts': ['Part A: Regex Fundamentals (Foundation) ',
                         'Part B: Pattern-Based Extraction (using '
                         're.findall()) ']},
 'Heading 2': {'count': 10,
               'texts': ['A2. Character Classes ', 'A3. Quantifiers ']},
 'Normal': {'count': 64,
            'texts': ['A1. What is Regex? ',
                      'Regex (Regular Expression) is a special pattern '
                      'language. It is used to find specific pieces of text, '
                      'such as emails, phone numbers, or dates, inside a big '
                      'block of text. ']}}


### **TASK 5**

In [31]:
from docx import Document

doc = Document("sample.docx")

tables_data = []

for table in doc.tables:
    table_data = []
    
    for row in table.rows:
        table_data.append([cell.text for cell in row.cells])
    
    tables_data.append(table_data)

print("Total tables:", len(tables_data))

for i, table in enumerate(tables_data, 1):
    rows = len(table)
    cols = len(table[0]) if table else 0
    print(f"Table {i}: {rows} × {cols}")


Total tables: 5
Table 1: 6 × 2
Table 2: 5 × 3
Table 3: 5 × 3
Table 4: 5 × 2
Table 5: 6 × 3


In [32]:
# (b)
import pandas as pd

first_table = tables_data[0]

df = pd.DataFrame(
    first_table[1:],
    columns=first_table[0]
)

print(df)

empty_cells = df.isna().sum()

# Also count empty strings
empty_cells += (df == "").sum()

print("\nEmpty cells per column:")
print(empty_cells)

     Symbol                                            Meaning 
0        \d                                    Any digit (0-9) 
1        \w   Any word character (letters, digits, underscore) 
2        \s                            Space, tab, or new line 
3     [a-z]                   Any lowercase letter from a to z 
4  [A-Z0-9]           A custom set - capital letters or digits 

Empty cells per column:
Symbol      0
Meaning     0
dtype: int64


In [33]:
# (c)
from docx.table import Table
from docx.text.paragraph import Paragraph

pre_table_paragraphs = []

for element in doc.element.body:
    if element.tag.endswith("tbl"):
        break
    
    if element.tag.endswith("p"):
        paragraph = Paragraph(element, doc)
        pre_table_paragraphs.append(paragraph.text)

print("Pre-table paragraphs:", len(pre_table_paragraphs))
print(pre_table_paragraphs)


Pre-table paragraphs: 7
['Part A: Regex Fundamentals (Foundation) ', 'A1. What is Regex? ', 'Regex (Regular Expression) is a special pattern language. It is used to find specific pieces of text, such as emails, phone numbers, or dates, inside a big block of text. ', 'Why it matters: Raw text (emails, chats, documents) contains useful information mixed with a lot of other text. Without regex, we would have to search everything by hand, which is slow and full of mistakes. ', 'Where it is used in NLP pipeline: Regex is used in the preprocessing step. Before any NLP model works on the data, we clean the text or pull out specific information (like emails or phone numbers) using regex. ', 'A2. Character Classes ', 'Character classes tell regex what type of character to look for. ']


In [34]:
# (d)
with open("docx_extracted.txt", "w", encoding="utf-8") as file:
    file.write("--- PRE-TABLE PARAGRAPHS ---\n")
    
    for paragraph in pre_table_paragraphs:
        file.write(paragraph + "\n")
    
    file.write("\n--- TABLE DATA ---\n")
    
    for row in first_table:
        file.write(" | ".join(row) + "\n")

print("File created: docx_extracted.txt")


File created: docx_extracted.txt
